# 001 B0 检索基线骨架（可执行占位流程）

目的：在本地小样本 / Kaggle 离线环境跑通 `test.parquet` 统计 + 占位 B0 全流程，
为后续手动实验准备 submission.csv 产物路径。不训练模型，不下载 train.parquet 与大权重。

- 输入 `test.parquet` 位置自动探测：本地 `data/raw/test.parquet`，Kaggle 侧搜索 `/kaggle/input/**/test.parquet`。
- 分子集合从数据动态读取，不硬编码 400 个 ID。
- B0 候选为 `CCO` 占位（与 sample_submission 占位语义对齐），只验证格式与管线，不代表基线分数。
- 输出：Kaggle 侧 `/kaggle/working/submission.csv`；本地侧 `artifacts/001-baseline/submission.csv` + `metrics.json`（均已在 .gitignore，不入仓）。
- 手动运行：本地用项目 `.venv` 按序执行各 cell；Kaggle 侧经 CLI 人工 push（本轮不 push）。

In [ ]:
import sys
from pathlib import Path

# 仓库根目录 bootstrap：本地( cwd 或其父目录 )与 Kaggle(/kaggle/working) 兼容
def find_repo_root() -> Path:
    cands = [Path.cwd(), Path.cwd().parent, Path("/kaggle/working")]
    for c in cands:
        if (c / "src" / "data.py").exists():
            return c
    # Kaggle 自包含 fallback：notebook 同目录即工作目录
    return Path.cwd()

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("ROOT =", ROOT)

from src import data as D
from src import retrieval as R
from src import evaluation as E
print("src imports OK")

In [ ]:
# 定位 test.parquet：本地优先，Kaggle input 次之（glob 搜索，不硬编码 dataset 名）
def find_test_parquet(root: Path) -> Path:
    local = root / "data" / "raw" / "test.parquet"
    if local.exists():
        return local
    kroot = Path("/kaggle/input")
    if kroot.exists():
        hits = sorted(kroot.glob("**/test.parquet"))
        if hits:
            return hits[0]
    raise FileNotFoundError("test.parquet not found (data/raw/test.parquet, /kaggle/input/**/test.parquet)")

TEST = find_test_parquet(ROOT)
print("TEST =", TEST, f"({TEST.stat().st_size} bytes)")

ov = D.parquet_overview(str(TEST))
print("rows =", ov["num_rows"], "row_groups =", ov["num_row_groups"])
print("columns =", ov["columns"])
print("id columns =", D.resolve_id_columns(str(TEST)))

In [ ]:
# 标量列分布：流式按列读取，不碰 ms2_* 大数组列
adduct = D.adduct_counts(str(TEST))
instr = D.instrument_counts(str(TEST))
ion = D.ionization_counts(str(TEST))
print("adduct:", dict(adduct))
print("instrument:", dict(instr))
print("ionization:", dict(ion))

In [ ]:
# ms2 长度配对检查：默认只查第 1 个 batch（本地小样本）；全量扫描放远端
chk = D.check_ms2_pairing(str(TEST), max_batches=1)
print(chk)
print("sample molecule_ids:", D.sample_molecule_ids(str(TEST), 5))

# adduct->中性质量占位调用示例（单值演示，不代表真实质量修正）
nm, ppm = R.adduct_to_neutral_mass(300.1234, "[M+H]+", ppm_window=20.0)
print("neutral placeholder =", nm, "ppm_window =", ppm)

# aggregate_by_molecule 接口演示（合成小例子；真实检索证据待 B1 接入）
demo = R.aggregate_by_molecule({"s2": ["CC", "CCO"], "s1": ["CCO", "CCC"]}, {"s1": "m1", "s2": "m1"})
print("aggregate demo =", demo)

In [ ]:
# 占位 B0：每分子写 25 个 CCO 占位（与 sample_submission 形状一致），走通提交管线
import csv
import json
from collections import Counter

# 读 spectrum_id -> molecule_id 映射（仅两列标量，小文件可流式全读）
spec2mol = {}
for batch in D.iter_batches(str(TEST), columns=["spectrum_id", "molecule_id"], batch_size=1024):
    sids = batch.column(0).to_pylist()
    mids = batch.column(1).to_pylist()
    for s, m in zip(sids, mids):
        spec2mol[s] = m

molecules = sorted(set(spec2mol.values()))
per_mol_counts = Counter(spec2mol.values())
print("n_spectra =", len(spec2mol), "n_molecules =", len(molecules))

# 占位候选：每分子与其谱数相关（记录用），内容恒为 25 个 CCO；
# 注意：真实检索聚合（去重保序，见 aggregate_by_molecule）在有检索信号后才会替代这里。
per_mol = {m: R.b0_placeholder_candidates(n_spectra=per_mol_counts[m]) for m in molecules}
assert set(per_mol) == set(molecules), "molecule coverage mismatch"

# 输出路径：Kaggle 侧固定 /kaggle/working/submission.csv；本地侧 artifacts（不入仓）
if Path("/kaggle/working").exists():
    out_csv = Path("/kaggle/working/submission.csv")
    out_metrics = Path("/kaggle/working/metrics.json")
else:
    adir = ROOT / "artifacts" / "001-baseline"
    adir.mkdir(parents=True, exist_ok=True)
    out_csv = adir / "submission.csv"
    out_metrics = adir / "metrics.json"

with open(out_csv, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["molecule_id", "smiles"])
    for m in molecules:
        w.writerow([m, ";".join(per_mol[m])])
print("wrote", out_csv)

# 格式自检：每行按官方语义拆分后非空项数 == 25（占位应全保留）
bad = sum(1 for m in molecules if len(E.split_candidates(";".join(per_mol[m]))) != 25)
metrics = {"n_molecules": len(molecules), "n_spectra": len(spec2mol), "rows_with_bad_width": bad,
           "note": "placeholder B0 only; no retrieval signal, no MRR claim"}
with open(out_metrics, "w") as f:
    json.dump(metrics, f, indent=2)
print(metrics)
assert bad == 0, f"{bad} rows with bad width"
print("NEXT: B1 analog/指纹重排与 B2 多谱融合；未见结构整组留出在 split manifest 做")